<h1 style="color: #ad4894;">Electric Vehicle Population Data Analysis and Insights</h1>

<h1 style="color: #cb9b43;">Introduction to Electric Vehicle Population Analysis</h1>

**Electric Vehicles (EVs)** play a critical role in reducing transportation emissions, lowering fossil fuel dependency, and accelerating the transition toward sustainable mobility. Understanding vehicle adoption patterns, battery range capabilities, manufacturer dominance, and regional concentration helps policymakers, utilities, and automakers plan charging infrastructure and clean-energy incentives effectively.

The purpose of this analysis is to explore and understand the key factors associated with Electric Vehicle adoption and present actionable insights through structured data preprocessing and tabular exploratory analysis. This project uses Python libraries such as **Pandas**, **NumPy**, **Matplotlib**, and **Seaborn** to clean, validate, and analyze the Electric Vehicle Population Dataset (`export.csv`).

<h1 style="color: #4FC3F7;">Data Source</h1>

The dataset used in this project is the **Electric Vehicle Population Data (`export.csv`)**, comprising Battery Electric Vehicles (BEVs) and Plug-in Hybrid Electric Vehicles (PHEVs) registered primarily through the Washington State Department of Licensing (DOL). It contains **300,142 vehicle registration records** across **16 attributes** spanning model years **1999 to 2027**.

**Note:** Two additional calculated features—**Vehicle_Age** (relative to the dataset's maximum model year `2027`) and **Range_Reported_Flag** (`Reported` vs. `Unresearched/Zero`)—are derived during preprocessing to enhance the analytical depth of the dataset.

<h1 style="color: #4FC3F7;">Attribute Details</h1>

| Attribute Name | Description |
| :--- | :--- |
| **VIN_Prefix** | First 10 characters of the Vehicle Identification Number (`VIN (1-10)`). |
| **County** | County in which the registered owner resides. |
| **City** | City in which the registered owner resides. |
| **State** | Two-letter state or territory code of the registration address (predominantly `WA`). |
| **Postal_Code** | 5-digit ZIP/postal code of the registered vehicle location. |
| **Model_Year** | Manufacturing model year of the electric vehicle (`1999`–`2027`). |
| **Make** | Manufacturer brand of the vehicle (e.g., `TESLA`, `CHEVROLET`, `FORD`, `NISSAN`). |
| **Model** | Specific vehicle model name (e.g., `MODEL 3`, `MODEL Y`, `LEAF`, `MUSTANG MACH-E`). |
| **EV_Type** | Electric vehicle propulsion category (`Battery Electric Vehicle (BEV)` or `Plug-in Hybrid Electric Vehicle (PHEV)`). |
| **CAFV_Eligibility** | Clean Alternative Fuel Vehicle (CAFV) eligibility classification based on battery range. |
| **Electric_Range** | Estimated all-electric driving range in miles (`0` indicates range not yet researched for newer models). |
| **Legislative_District** | Washington State legislative district number (`1`–`49`, `0` for out-of-state registrations). |
| **DOL_Vehicle_ID** | Unique numerical identifier assigned to each vehicle by the Department of Licensing. |
| **Vehicle_Location** | Geographic coordinates (`POINT (Longitude Latitude)`) of the postal centroid. |
| **Electric_Utility** | Electric power utility provider(s) serving the vehicle's registered location. |
| **Census_Tract_2020** | 2020 U.S. Census Tract GEOID identifier. |
| **Vehicle_Age** | Derived feature calculated as `2027 - Model_Year` (in years). |
| **Range_Status** | Derived indicator distinguishing vehicles with reported electric range (`> 0` miles) from unresearched (`0` miles). |

<div style="background-color: #1E3A5F; color: #FFFFFF; padding: 10px 16px; border-left: 5px solid #4FC3F7; border-radius: 4px; font-size: 24px; font-weight: bold;">
  Importing Required Libraries
</div>

All necessary Python libraries such as **Pandas**, **NumPy**, **Matplotlib**, and **Seaborn** are imported for performing data loading, cleaning, exploration, and visualization.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

<div style="background-color: #1E3A5F; color: #FFFFFF; padding: 10px 16px; border-left: 5px solid #4FC3F7; border-radius: 4px; font-size: 24px; font-weight: bold;">
  Data Loading and Initial Overview
</div>

This step focuses on importing the Electric Vehicle Population Dataset (`export.csv`) and inspecting its initial structure before starting the data cleaning phase.

### (i) Loading the Dataset

In [4]:
df = pd.read_csv("export.csv", low_memory=False)
df.head()

,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 GEOID
0,3FMTK3SU8P,King,Seattle,WA,98177.0,2023,FORD,MUSTANG MACH-E,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,32.0,254622924,POINT (-122.37238 47.74068),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303300e+10
1,JM3KKEHA8S,Thurston,Yelm,WA,98597.0,2025,MAZDA,CX-90,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,26.0,2.0,277791643,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,5.306701e+10
2,5YJ3E1EB8J,King,Sammamish,WA,98075.0,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215.0,41.0,347418676,POINT (-122.03539 47.61344),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303303e+10
3,JHMZC5F12K,Island,Oak Harbor,WA,98277.0,2019,HONDA,CLARITY,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,47.0,10.0,223620701,POINT (-122.64682 48.29077),PUGET SOUND ENERGY INC,5.302997e+10
4,1C4RJXN65R,Kitsap,Bremerton,WA,98312.0,2024,JEEP,WRANGLER,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,21.0,26.0,256402674,POINT (-122.65223 47.57192),PUGET SOUND ENERGY INC,5.303508e+10


### (ii) Checking Dataset Dimensions

Inspecting the total number of rows and columns in the raw dataset.

In [5]:
df.shape

(300142, 16)

<div style="background-color: #1E3A5F; color: #FFFFFF; padding: 10px 16px; border-left: 5px solid #4FC3F7; border-radius: 4px; font-size: 24px; font-weight: bold;">
  Data cleaning and processing
</div>

This phase addresses data quality issues including duplicate records, missing values, incorrect data types, formatting inconsistencies, outliers, and logical discrepancies.

### Step 1: Remove Duplicate Records

Checking for and removing duplicate vehicle registration records (both full-row duplicates and duplicate `DOL Vehicle ID` entries) to ensure data integrity and prevent skewed analysis.

In [7]:
# Check the number of duplicate records
print("Duplicate rows before removal:", df.duplicated().sum())
print("Duplicate DOL Vehicle IDs before removal:", df.duplicated(subset=['DOL Vehicle ID']).sum())


Duplicate rows before removal: 0
Duplicate DOL Vehicle IDs before removal: 0


In [8]:
# Remove duplicate records if any exist
df.drop_duplicates(inplace=True)
df.drop_duplicates(subset=['DOL Vehicle ID'], keep='first', inplace=True)


In [10]:
# Verify removal and check the updated dataset shape
print("Duplicate rows after removal:", df.duplicated().sum())
print("Updated dataset shape:", df.shape)

Duplicate rows after removal: 0
Updated dataset shape: (300142, 16)


### Step 2: Handle Missing Values (NULLs)

Identifying missing values across all columns and applying domain-appropriate imputation:
* Numeric missing values in `Electric Range` (35 records) are imputed using the group median by `Make`, `Model`, and `Model Year` (falling back to `0.0`).
* Missing `Legislative District` values (790 records, primarily out-of-state registrations) and `Postal Code` / `2020 GEOID` values (15 records) are filled with `0` to denote non-applicable/out-of-state entries.
* Missing categorical fields (`County`, `City`, `Vehicle Location`, `Electric Utility`) are imputed with `'Unknown'`.

In [11]:
# Check for missing values in each column
print("Missing values before handling:\n", df.isnull().sum())

Missing values before handling:
 VIN (1-10)                                             0
County                                                15
City                                                  15
State                                                  0
Postal Code                                           15
Model Year                                             0
Make                                                   0
Model                                                  0
Electric Vehicle Type                                  0
Clean Alternative Fuel Vehicle (CAFV) Eligibility      0
Electric Range                                        35
Legislative District                                 790
DOL Vehicle ID                                         0
Vehicle Location                                      24
Electric Utility                                      15
2020 GEOID                                            15
dtype: int64


In [12]:
# Impute missing 'Electric Range' using group median by Make, Model, Model Year, then fallback to 0
df['Electric Range'] = df['Electric Range'].fillna(
    df.groupby(['Make', 'Model', 'Model Year'])['Electric Range'].transform('median')
).fillna(0.0)

In [13]:
# Fill missing numeric geographic identifiers with 0 (out-of-state / unmapped)
df['Legislative District'] = df['Legislative District'].fillna(0)
df['Postal Code'] = df['Postal Code'].fillna(0)
df['2020 GEOID'] = df['2020 GEOID'].fillna(0)

In [14]:
# Fill missing categorical text columns with 'Unknown'
for col in ['County', 'City', 'Vehicle Location', 'Electric Utility']:
    df[col] = df[col].fillna('Unknown')

In [15]:
# Verify that all missing values are handled
print("\nMissing values after handling:\n", df.isnull().sum())


Missing values after handling:
 VIN (1-10)                                           0
County                                               0
City                                                 0
State                                                0
Postal Code                                          0
Model Year                                           0
Make                                                 0
Model                                                0
Electric Vehicle Type                                0
Clean Alternative Fuel Vehicle (CAFV) Eligibility    0
Electric Range                                       0
Legislative District                                 0
DOL Vehicle ID                                       0
Vehicle Location                                     0
Electric Utility                                     0
2020 GEOID                                           0
dtype: int64


### Step 3: Clean Column Spaces and Correct Data Types

Stripping leading/trailing spaces from column headers and converting columns (`Model Year`, `Postal Code`, `Legislative District`, `DOL Vehicle ID`, `2020 GEOID`, `Electric Vehicle Type`, `State`) into their appropriate integer or categorical data types for efficient memory usage and processing.

In [16]:
# Remove any leading or trailing spaces from column names
df.columns = df.columns.str.strip()

In [17]:
# Check current data types of all columns
print("Data types before correction:\n", df.dtypes)

Data types before correction:
 VIN (1-10)                                               str
County                                                   str
City                                                     str
State                                                    str
Postal Code                                          float64
Model Year                                             int64
Make                                                     str
Model                                                    str
Electric Vehicle Type                                    str
Clean Alternative Fuel Vehicle (CAFV) Eligibility        str
Electric Range                                       float64
Legislative District                                 float64
DOL Vehicle ID                                         int64
Vehicle Location                                         str
Electric Utility                                         str
2020 GEOID                                           f

In [18]:
# Ensure appropriate data types
df['Model Year'] = df['Model Year'].astype(int)
df['Postal Code'] = df['Postal Code'].astype('int64')
df['Legislative District'] = df['Legislative District'].astype(int)
df['DOL Vehicle ID'] = df['DOL Vehicle ID'].astype('int64')
df['2020 GEOID'] = df['2020 GEOID'].astype('int64')
df['Electric Range'] = df['Electric Range'].astype(float)
df['Electric Vehicle Type'] = df['Electric Vehicle Type'].astype('category')
df['State'] = df['State'].astype('category')

In [19]:
# Verify updated data types
print("\nData types after correction:\n", df.dtypes)


Data types after correction:
 VIN (1-10)                                                str
County                                                    str
City                                                      str
State                                                category
Postal Code                                             int64
Model Year                                              int64
Make                                                      str
Model                                                     str
Electric Vehicle Type                                category
Clean Alternative Fuel Vehicle (CAFV) Eligibility         str
Electric Range                                        float64
Legislative District                                    int64
DOL Vehicle ID                                          int64
Vehicle Location                                          str
Electric Utility                                          str
2020 GEOID                             

### Step 4: Check and Remove Irrelevant Columns

Verifying all dataset features to ensure no unnamed or irrelevant columns are present in the DataFrame.

In [20]:
# Check all existing columns
print("Current columns:", df.columns.tolist())

Current columns: ['VIN (1-10)', 'County', 'City', 'State', 'Postal Code', 'Model Year', 'Make', 'Model', 'Electric Vehicle Type', 'Clean Alternative Fuel Vehicle (CAFV) Eligibility', 'Electric Range', 'Legislative District', 'DOL Vehicle ID', 'Vehicle Location', 'Electric Utility', '2020 GEOID']


In [21]:
# Drop any unnamed or irrelevant columns if present
irrelevant_cols = [col for col in df.columns if 'Unnamed' in col]
df.drop(columns=irrelevant_cols, inplace=True)

In [22]:
# Verify remaining columns
print("Irrelevant columns removed:", irrelevant_cols)
print("Remaining columns count:", df.shape[1])

Irrelevant columns removed: []
Remaining columns count: 16


### Step 5: Standardize Text Formats and Correct Formatting Inconsistencies

Standardizing text values across categorical columns (`Make`, `Model`, `County`, `City`, `State`, `Electric Vehicle Type`, `Clean Alternative Fuel Vehicle (CAFV) Eligibility`) by stripping extra whitespace and ensuring consistent casing.

In [23]:
# Check unique values in 'Electric Vehicle Type' and 'State' before standardization
print("Unique EV Types before cleaning:", df['Electric Vehicle Type'].unique().tolist())

Unique EV Types before cleaning: ['Battery Electric Vehicle (BEV)', 'Plug-in Hybrid Electric Vehicle (PHEV)']


In [24]:
# Standardize text formatting across key categorical columns
df['Make'] = df['Make'].astype(str).str.strip().str.upper()
df['Model'] = df['Model'].astype(str).str.strip().str.upper()
df['County'] = df['County'].astype(str).str.strip().str.title()
df['City'] = df['City'].astype(str).str.strip().str.title()
df['State'] = df['State'].astype(str).str.strip().str.upper().astype('category')
df['Electric Vehicle Type'] = df['Electric Vehicle Type'].astype(str).str.strip().astype('category')
df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'] = (
    df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].astype(str).str.strip().astype('category')
)

In [25]:
# Verify cleaned values
print("Unique EV Types after cleaning:", df['Electric Vehicle Type'].unique().tolist())
print("Top 5 Makes after standardization:", df['Make'].value_counts().head(5).to_dict())

Unique EV Types after cleaning: ['Battery Electric Vehicle (BEV)', 'Plug-in Hybrid Electric Vehicle (PHEV)']
Top 5 Makes after standardization: {'TESLA': 122291, 'CHEVROLET': 20218, 'FORD': 15997, 'NISSAN': 15994, 'KIA': 14897}


### Step 6: Detect and Handle Outliers

Evaluating numerical distributions (`Model Year` and non-zero `Electric Range`) using the **Interquartile Range (IQR)** method. Because `Electric Range = 0` represents unresearched battery ranges rather than physical zero-mile vehicles, IQR bounds are inspected specifically on reported positive ranges (`Electric Range > 0`) and `Model Year` to preserve legitimate long-range EV records without distortion.

In [26]:
# Check IQR outlier thresholds for Model Year and reported Electric Range (> 0)
for col in ['Model Year', 'Electric Range']:
    series = df.loc[df['Electric Range'] > 0, col] if col == 'Electric Range' else df[col]
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    
    outliers_count = ((series < lower_limit) | (series > upper_limit)).sum()
    print(f"{col}: {outliers_count} statistical outliers detected (Bounds: {lower_limit:.2f} to {upper_limit:.2f})")

Model Year: 8710 statistical outliers detected (Bounds: 2015.00 to 2031.00)
Electric Range: 0 statistical outliers detected (Bounds: -247.50 to 492.50)


In [27]:
# Cap extreme upper outliers on Electric Range if any exceed the upper IQR threshold
valid_range = df.loc[df['Electric Range'] > 0, 'Electric Range']
q1_r, q3_r = valid_range.quantile(0.25), valid_range.quantile(0.75)
upper_range_cap = q3_r + 1.5 * (q3_r - q1_r)
df['Electric Range'] = np.where(df['Electric Range'] > upper_range_cap, upper_range_cap, df['Electric Range'])

In [28]:
# Verify summary statistics after outlier review
df[['Model Year', 'Electric Range']].describe().round(2)

,Model Year,Electric Range
count,300142.00,300142.00
mean,2022.39,36.14
std,3.10,75.44
min,1999.00,0.00
25%,2021.00,0.00
50%,2023.00,0.00
75%,2025.00,32.00
max,2027.00,337.00


### Step 7: Validate Data Accuracy and Create Derived Features

Validating logical constraints (valid model year bounds, non-negative electric ranges, valid EV categories) and engineering two derived features:
1. **`Vehicle_Age`**: Calculated relative to the dataset's maximum model year (`2027 - Model Year`).
2. **`Range_Status`**: Categorizes records into `'Reported (>0 mi)'` vs. `'Unresearched (0 mi)'`.

In [29]:
# Validate Model Year range (must be between 1990 and 2027)
invalid_year = df[(df['Model Year'] < 1990) | (df['Model Year'] > 2027)]
print("Invalid Model Year records:", len(invalid_year))

Invalid Model Year records: 0


In [30]:
# Validate Electric Vehicle Type categories
print("Unique values in Electric Vehicle Type:", df['Electric Vehicle Type'].unique().tolist())

Unique values in Electric Vehicle Type: ['Battery Electric Vehicle (BEV)', 'Plug-in Hybrid Electric Vehicle (PHEV)']


In [31]:
# Create derived analytical features
max_year = df['Model Year'].max()
df['Vehicle_Age'] = (max_year - df['Model Year']).astype(int)
df['Range_Status'] = np.where(df['Electric Range'] > 0, 'Reported (>0 mi)', 'Unresearched (0 mi)')

In [32]:
# Verify no negative values exist in key numerical columns
num_check_cols = ['Model Year', 'Electric Range', 'Vehicle_Age', 'Postal Code', 'Legislative District']
print("Negative values count per column:\n", (df[num_check_cols] < 0).sum())

Negative values count per column:
 Model Year              0
Electric Range          0
Vehicle_Age             0
Postal Code             0
Legislative District    0
dtype: int64


### Step 8: Resolve Discrepancies, Rename Columns, and Export Cleaned Data

Standardizing column names with clean `snake_case` identifiers for analytical clarity and saving the final cleaned dataset as `cleaned_ev_population_dataset.csv`.

In [33]:
# Rename columns for clarity and consistency
df.rename(columns={
    'VIN (1-10)': 'VIN_Prefix',
    'Postal Code': 'Postal_Code',
    'Model Year': 'Model_Year',
    'Electric Vehicle Type': 'EV_Type',
    'Clean Alternative Fuel Vehicle (CAFV) Eligibility': 'CAFV_Eligibility',
    'Electric Range': 'Electric_Range',
    'Legislative District': 'Legislative_District',
    'DOL Vehicle ID': 'DOL_Vehicle_ID',
    'Vehicle Location': 'Vehicle_Location',
    'Electric Utility': 'Electric_Utility',
    '2020 GEOID': 'Census_Tract_2020'
}, inplace=True)

In [34]:
# Save the cleaned dataset to a new CSV file
df.to_csv("cleaned_ev_population_dataset.csv", index=False)

In [35]:
# Display the final cleaned dataset overview
print("Data cleaning completed successfully! Final shape:", df.shape)
df.head()

Data cleaning completed successfully! Final shape: (300142, 18)


,VIN_Prefix,County,City,State,Postal_Code,Model_Year,Make,Model,EV_Type,CAFV_Eligibility,Electric_Range,Legislative_District,DOL_Vehicle_ID,Vehicle_Location,Electric_Utility,Census_Tract_2020,Vehicle_Age,Range_Status
0,3FMTK3SU8P,King,Seattle,WA,98177,2023,FORD,MUSTANG MACH-E,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,32,254622924,POINT (-122.37238 47.74068),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),53033000402,4,Unresearched (0 mi)
1,JM3KKEHA8S,Thurston,Yelm,WA,98597,2025,MAZDA,CX-90,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,26.0,2,277791643,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,53067012412,2,Reported (>0 mi)
2,5YJ3E1EB8J,King,Sammamish,WA,98075,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215.0,41,347418676,POINT (-122.03539 47.61344),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),53033032224,9,Reported (>0 mi)
3,JHMZC5F12K,Island,Oak Harbor,WA,98277,2019,HONDA,CLARITY,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,47.0,10,223620701,POINT (-122.64682 48.29077),PUGET SOUND ENERGY INC,53029970800,8,Reported (>0 mi)
4,1C4RJXN65R,Kitsap,Bremerton,WA,98312,2024,JEEP,WRANGLER,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,21.0,26,256402674,POINT (-122.65223 47.57192),PUGET SOUND ENERGY INC,53035080600,3,Reported (>0 mi)


✔ **Data Preprocessing was successfully completed!!** Ensuring the dataset is clean, consistent, and ready for analysis.

---

# Exploratory Data Analysis (EDA)

**Exploratory Data Analysis (EDA)** is the process of summarizing, visualizing, and understanding the underlying patterns, trends, relationships, and distributions within the dataset before modeling or decision making.

### (i) Overview of the Dataset

Examine the dataset's size, structure, and data types using `df.shape`, `df.head()`, and `df.info()` to get an initial understanding of the cleaned data.

In [36]:
# Displaying the Dimensions of the Dataset
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

Number of Rows: 300142
Number of Columns: 18


In [37]:
# Displaying the First 5 Rows
df.head()

,VIN_Prefix,County,City,State,Postal_Code,Model_Year,Make,Model,EV_Type,CAFV_Eligibility,Electric_Range,Legislative_District,DOL_Vehicle_ID,Vehicle_Location,Electric_Utility,Census_Tract_2020,Vehicle_Age,Range_Status
0,3FMTK3SU8P,King,Seattle,WA,98177,2023,FORD,MUSTANG MACH-E,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,32,254622924,POINT (-122.37238 47.74068),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),53033000402,4,Unresearched (0 mi)
1,JM3KKEHA8S,Thurston,Yelm,WA,98597,2025,MAZDA,CX-90,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,26.0,2,277791643,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,53067012412,2,Reported (>0 mi)
2,5YJ3E1EB8J,King,Sammamish,WA,98075,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215.0,41,347418676,POINT (-122.03539 47.61344),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),53033032224,9,Reported (>0 mi)
3,JHMZC5F12K,Island,Oak Harbor,WA,98277,2019,HONDA,CLARITY,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,47.0,10,223620701,POINT (-122.64682 48.29077),PUGET SOUND ENERGY INC,53029970800,8,Reported (>0 mi)
4,1C4RJXN65R,Kitsap,Bremerton,WA,98312,2024,JEEP,WRANGLER,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,21.0,26,256402674,POINT (-122.65223 47.57192),PUGET SOUND ENERGY INC,53035080600,3,Reported (>0 mi)


In [38]:
# Displaying Dataset Information (Columns, Data Types, Non-Null Counts)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300142 entries, 0 to 300141
Data columns (total 18 columns):
 #   Column                Non-Null Count   Dtype   
---  ------                --------------   -----   
 0   VIN_Prefix            300142 non-null  str     
 1   County                300142 non-null  str     
 2   City                  300142 non-null  str     
 3   State                 300142 non-null  category
 4   Postal_Code           300142 non-null  int64   
 5   Model_Year            300142 non-null  int64   
 6   Make                  300142 non-null  str     
 7   Model                 300142 non-null  str     
 8   EV_Type               300142 non-null  category
 9   CAFV_Eligibility      300142 non-null  category
 10  Electric_Range        300142 non-null  float64 
 11  Legislative_District  300142 non-null  int64   
 12  DOL_Vehicle_ID        300142 non-null  int64   
 13  Vehicle_Location      300142 non-null  str     
 14  Electric_Utility      300142 non-null  str     

**Observation:**
* The cleaned dataset consists of **300,142 vehicle registration records** and **18 attributes** (16 original features plus 2 derived analytical features: `Vehicle_Age` and `Range_Status`).
* There are **no missing (null) values** across any columns, and all features are stored in their appropriate numerical, string, or categorical data types, confirming the dataset is ready for statistical analysis.

### (ii) Statistical Summary

Analyzing the central tendency, dispersion, and distribution of all numerical features using `df.describe()`.

In [39]:
# Displaying summary statistics for numerical columns
df[['Model_Year', 'Electric_Range', 'Vehicle_Age', 'Legislative_District']].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Model_Year,300142.0,2022.39,3.10,1999.0,2021.0,2023.0,2025.0,2027.0
Electric_Range,300142.0,36.14,75.44,0.0,0.0,0.0,32.0,337.0
Vehicle_Age,300142.0,4.61,3.10,0.0,2.0,4.0,6.0,28.0
Legislative_District,300142.0,28.64,14.94,0.0,17.0,32.0,42.0,49.0


**Observation:**
* **Model Year & Vehicle Age:** Vehicle model years span from **1999 to 2027**, with a median model year in the mid-2020s, indicating rapid recent EV fleet growth.
* **Electric Range:** Overall mean electric range is pulled downward because **197,085 records (~65.7%)** have a recorded value of `0` (representing newer BEVs whose battery range has not yet been tested/researched by EPA/DOL), whereas researched vehicles reach up to **337 miles**.
* **Legislative District:** Values span districts `1` through `49` across Washington State (`0` denotes out-of-state registrations).

### (iii) Target Variable (`EV_Type`) & `CAFV_Eligibility` Distribution

Examining the exact counts and percentage breakdown of the primary vehicle classification (`EV_Type`) and clean fuel qualification status (`CAFV_Eligibility`).

In [40]:
# Count and percentage of Electric Vehicle Type (BEV vs. PHEV)
print("--- Electric Vehicle Type Distribution ---")
ev_counts = df['EV_Type'].value_counts()
ev_pct = (df['EV_Type'].value_counts(normalize=True) * 100).round(2)
print(pd.DataFrame({'Count': ev_counts, 'Percentage (%)': ev_pct}))

--- Electric Vehicle Type Distribution ---
                                         Count  Percentage (%)
EV_Type                                                       
Battery Electric Vehicle (BEV)          241840           80.58
Plug-in Hybrid Electric Vehicle (PHEV)   58302           19.42


In [41]:
print("\n--- CAFV Eligibility Distribution ---")
cafv_counts = df['CAFV_Eligibility'].value_counts()
cafv_pct = (df['CAFV_Eligibility'].value_counts(normalize=True) * 100).round(2)
print(pd.DataFrame({'Count': cafv_counts, 'Percentage (%)': cafv_pct}))


--- CAFV Eligibility Distribution ---
                                                     Count  Percentage (%)
CAFV_Eligibility                                                          
Eligibility unknown as battery range has not be...  197085           65.66
Clean Alternative Fuel Vehicle Eligible              78933           26.30
Not eligible due to low battery range                24124            8.04


**Observation:**
* **BEV Dominance:** **Battery Electric Vehicles (BEVs)** account for **80.58%** (241,840 vehicles) of the fleet, outnumbering **Plug-in Hybrid Electric Vehicles (PHEVs)** at **19.42%** (58,302 vehicles) by more than 4-to-1.
* **CAFV Eligibility Breakdown:** Roughly **65.66%** fall under *"Eligibility unknown as battery range has not been researched"*, **26.30%** are *"Clean Alternative Fuel Vehicle Eligible"*, and **8.04%** are *"Not eligible due to low battery range"*.

### (iv) Comparative Analysis by Vehicle Type (`EV_Type`)

Comparing average vehicle metrics between **Battery Electric Vehicles (BEVs)** and **Plug-in Hybrid Electric Vehicles (PHEVs)**, along with a cross-tabulation of Top Manufacturers (`Make`) vs. `EV_Type`.

In [42]:
# Cross-tabulation: Top 10 Makes vs Electric Vehicle Type
top_10_makes = df['Make'].value_counts().head(10).index
print("--- Top 10 Makes vs. Electric Vehicle Type ---")
print(pd.crosstab(df.loc[df['Make'].isin(top_10_makes), 'Make'], df['EV_Type'], margins=True))

--- Top 10 Makes vs. Electric Vehicle Type ---
EV_Type    Battery Electric Vehicle (BEV)  \
Make                                        
BMW                                  5477   
CHEVROLET                           15848   
FORD                                12163   
HYUNDAI                             11201   
KIA                                 10440   
NISSAN                              15962   
RIVIAN                              10150   
TESLA                              122291   
TOYOTA                               3359   
VOLVO                                2906   
All                                209797   

EV_Type    Plug-in Hybrid Electric Vehicle (PHEV)     All  
Make                                                       
BMW                                          6918   12395  
CHEVROLET                                    4370   20218  
FORD                                         3834   15997  
HYUNDAI                                      1380   12581  
KIA    

In [43]:
# Mean numerical values grouped by EV_Type (overall and for vehicles with reported range > 0)
print("\n--- Average Vehicle Metrics by EV Type (All Records) ---")
display(df.groupby('EV_Type', observed=False)[['Model_Year', 'Vehicle_Age', 'Electric_Range']].mean().T.round(2))

print("\n--- Average Electric Range by EV Type (Reported Range > 0 Only) ---")
df[df['Electric_Range'] > 0].groupby('EV_Type', observed=False)[['Electric_Range', 'Model_Year']].mean().T.round(2)


--- Average Vehicle Metrics by EV Type (All Records) ---


EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
Model_Year,2022.62,2021.43
Vehicle_Age,4.38,5.57
Electric_Range,37.08,32.26



--- Average Electric Range by EV Type (Reported Range > 0 Only) ---


EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
Electric_Range,200.37,32.28
Model_Year,2017.68,2021.43


**Observation:**
* **Manufacturer Specialization:** **TESLA** (122,291 vehicles), **NISSAN**, and **RIVIAN** produce exclusively BEVs in this dataset, whereas brands like **TOYOTA**, **VOLVO**, **BMW**, and **FORD** maintain a substantial mix of both BEVs and PHEVs.
* **Electric Range Separation:** Among vehicles with reported battery ranges (`> 0` miles), BEVs deliver dramatically higher all-electric driving ranges than PHEVs, which typically operate in the 20–45 mile electric-only band before switching to hybrid gasoline power.

### (v) Correlation Matrix

Calculating the Pearson correlation coefficients between numerical features (`Model_Year`, `Electric_Range`, `Vehicle_Age`, `Postal_Code`, `Legislative_District`) to identify strong positive or negative relationships.

In [44]:
# Displaying the numerical correlation matrix
corr_cols = ['Model_Year', 'Vehicle_Age', 'Electric_Range', 'Postal_Code', 'Legislative_District']
df[corr_cols].corr().round(2)

,Model_Year,Vehicle_Age,Electric_Range,Postal_Code,Legislative_District
Model_Year,1.00,-1.00,-0.55,0.00,0.00
Vehicle_Age,-1.00,1.00,0.55,-0.00,-0.00
Electric_Range,-0.55,0.55,1.00,0.00,0.00
Postal_Code,0.00,-0.00,0.00,1.00,0.03
Legislative_District,0.00,-0.00,0.00,0.03,1.00


**Observation:**
* **Strong Negative Correlations:**
  * `Model_Year` and `Vehicle_Age` exhibit an exact `-1.00` inverse correlation by definition.
  * `Model_Year` and `Electric_Range` show a notable negative correlation because post-2020 BEVs are predominantly recorded with `0` miles pending range research, whereas older model years (2017–2020) retain explicit EPA range figures.
* **Weak Geographic Correlations:** `Legislative_District` and `Postal_Code` show minimal linear correlation with vehicle age or range, indicating EV adoption spans multiple price/range tiers across districts.

### (vi) Multivariate Analysis (Tabular)

Analyzing the combined effect of multiple variables simultaneously—specifically how **EV Type**, **Model Year Era**, and **Top Counties** together relate to fleet counts and average reported `Electric_Range`.

In [45]:
# Multivariate Table 1: Average metrics grouped by EV_Type and Range_Status
key_cols = ['Model_Year', 'Vehicle_Age', 'Electric_Range']
multi_table = df.groupby(['EV_Type', 'Range_Status'], observed=False)[key_cols].agg(['count', 'mean']).round(2)
multi_table

Model_Year  \
                                                                count   
EV_Type                                Range_Status                     
Battery Electric Vehicle (BEV)         Reported (>0 mi)         44755   
                                       Unresearched (0 mi)     197085   
Plug-in Hybrid Electric Vehicle (PHEV) Reported (>0 mi)         58267   
                                       Unresearched (0 mi)         35   

                                                                     \
                                                               mean   
EV_Type                                Range_Status                   
Battery Electric Vehicle (BEV)         Reported (>0 mi)     2017.68   
                                       Unresearched (0 mi)  2023.74   
Plug-in Hybrid Electric Vehicle (PHEV) Reported (>0 mi)     2021.43   
                                       Unresearched (0 mi)  2026.37   

                                                           Vehicle_Age        \
                                                                 count  mean   
EV_Type                                Range_Status                            
Battery Electric Vehicle (BEV)         Reported (>0 mi)          44755  9.32   
                                       Unresearched (0 mi)      197085  3.26   
Plug-in Hybrid Electric Vehicle (PHEV) Reported (>0 mi)          58267  5.57   
                                       Unresearched (0 mi)          35  0.63   

                                                           Electric_Range  \
                                                                    count   
EV_Type                                Range_Status                         
Battery Electric Vehicle (BEV)         Reported (>0 mi)             44755   
                                       Unresearched (0 mi)         197085   
Plug-in Hybrid Electric Vehicle (PHEV) Reported (>0 mi)             58267   
                                       Unresearched (0 mi)             35   

                                                                    
                                                              mean  
EV_Type                                Range_Status                 
Battery Electric Vehicle (BEV)         Reported (>0 mi)     200.37  
                                       Unresearched (0 mi)    0.00  
Plug-in Hybrid Electric Vehicle (PHEV) Reported (>0 mi)      32.28  
                                       Unresearched (0 mi)    0.00

In [46]:
# Multivariate Table 2: Pivot table of Model Year Era, EV_Type, and Top 5 Counties (Vehicle Counts)
year_eras = pd.cut(
    df['Model_Year'],
    bins=[1998, 2015, 2019, 2022, 2027],
    labels=['Early (1999-2015)', 'Growth (2016-2019)', 'Expansion (2020-2022)', 'Current (2023-2027)']
)
top_5_counties = df['County'].value_counts().head(5).index

pivot_multi = pd.pivot_table(
    df[df['County'].isin(top_5_counties)],
    values='DOL_Vehicle_ID',
    index=year_eras[df['County'].isin(top_5_counties)],
    columns=['County', 'EV_Type'],
    aggfunc='count',
    observed=False
)

print("--- Vehicle Registration Counts by Model Year Era, Top 5 Counties, and EV Type ---")
pivot_multi

--- Vehicle Registration Counts by Model Year Era, Top 5 Counties, and EV Type ---


County                                         Clark  \
EV_Type               Battery Electric Vehicle (BEV)   
Model_Year                                             
Early (1999-2015)                                496   
Growth (2016-2019)                              1524   
Expansion (2020-2022)                           3261   
Current (2023-2027)                             9318   

County                                                        \
EV_Type               Plug-in Hybrid Electric Vehicle (PHEV)   
Model_Year                                                     
Early (1999-2015)                                        365   
Growth (2016-2019)                                       797   
Expansion (2020-2022)                                    952   
Current (2023-2027)                                     2202   

County                                          King  \
EV_Type               Battery Electric Vehicle (BEV)   
Model_Year                                             
Early (1999-2015)                               3444   
Growth (2016-2019)                             12464   
Expansion (2020-2022)                          24616   
Current (2023-2027)                            78535   

County                                                        \
EV_Type               Plug-in Hybrid Electric Vehicle (PHEV)   
Model_Year                                                     
Early (1999-2015)                                       1748   
Growth (2016-2019)                                      4849   
Expansion (2020-2022)                                   4961   
Current (2023-2027)                                    13359   

County                                        Pierce  \
EV_Type               Battery Electric Vehicle (BEV)   
Model_Year                                             
Early (1999-2015)                                634   
Growth (2016-2019)                              2110   
Expansion (2020-2022)                           4171   
Current (2023-2027)                            12888   

County                                                        \
EV_Type               Plug-in Hybrid Electric Vehicle (PHEV)   
Model_Year                                                     
Early (1999-2015)                                        519   
Growth (2016-2019)                                      1137   
Expansion (2020-2022)                                    976   
Current (2023-2027)                                     2734   

County                                     Snohomish  \
EV_Type               Battery Electric Vehicle (BEV)   
Model_Year                                             
Early (1999-2015)                                900   
Growth (2016-2019)                              2977   
Expansion (2020-2022)                           6447   
Current (2023-2027)                            21564   

County                                                        \
EV_Type               Plug-in Hybrid Electric Vehicle (PHEV)   
Model_Year                                                     
Early (1999-2015)                                        557   
Growth (2016-2019)                                      1188   
Expansion (2020-2022)                                   1157   
Current (2023-2027)                                     3013   

County                                      Thurston  \
EV_Type               Battery Electric Vehicle (BEV)   
Model_Year                                             
Early (1999-2015)                                355   
Growth (2016-2019)                              1033   
Expansion (2020-2022)                           1734   
Current (2023-2027)                             5405   

County                                                        
EV_Type               Plug-in Hybrid Electric Vehicle (PHEV)  
Model_Year                                                    
Early (1999-2015)                                 

**Observation (Multivariate Analysis):**
* **Temporal Surge Across Counties:** Across all top Washington counties (**King**, **Snohomish**, **Pierce**, **Clark**, and **Thurston**), registrations accelerate sharply in the **Expansion (2020–2022)** and **Current (2023–2027)** eras, led overwhelmingly by BEVs in King County (143,976 total EVs).
* **Range Reporting Shift:** Almost all `Unresearched (0 mi)` records belong to newer BEVs (`2021–2027`), whereas PHEVs continue to have their electric range reported across recent model years.

### Bivariate Analysis (Tabular)

Bivariate analysis examines the relationship between **two variables simultaneously** to uncover patterns, group differences, and pairwise associations. Here, we analyze:
1. **Categorical vs. Target:** `CAFV_Eligibility` and `Model Year Era` against `EV_Type`.
2. **Numerical vs. Target:** Vehicle metric averages and medians across `EV_Type` groups.
3. **Numerical vs. Numerical:** Strongest pairwise feature correlations.

In [47]:
# 1. CAFV Eligibility vs EV Type (Counts & Row Percentages)
cafv_ct = pd.crosstab(df['CAFV_Eligibility'], df['EV_Type'], margins=True)
cafv_pct = pd.crosstab(df['CAFV_Eligibility'], df['EV_Type'], normalize='index').mul(100).round(2)

print("--- CAFV Eligibility vs. EV Type (Counts) ---")
print(cafv_ct)
print("\n--- CAFV Eligibility vs. EV Type (Row %) ---")
print(cafv_pct)

--- CAFV Eligibility vs. EV Type (Counts) ---
EV_Type                                             Battery Electric Vehicle (BEV)  \
CAFV_Eligibility                                                                     
Clean Alternative Fuel Vehicle Eligible                                      44745   
Eligibility unknown as battery range has not be...                          197085   
Not eligible due to low battery range                                           10   
All                                                                         241840   

EV_Type                                             Plug-in Hybrid Electric Vehicle (PHEV)  \
CAFV_Eligibility                                                                             
Clean Alternative Fuel Vehicle Eligible                                              34188   
Eligibility unknown as battery range has not be...                                       0   
Not eligible due to low battery range                        

In [48]:
# 2. Model Year Era vs EV Type (Row Percentages)
era_bins = pd.cut(
    df['Model_Year'],
    bins=[1998, 2015, 2019, 2022, 2027],
    labels=['Early (1999-2015)', 'Growth (2016-2019)', 'Expansion (2020-2022)', 'Current (2023-2027)']
)
era_pct = pd.crosstab(era_bins, df['EV_Type'], normalize='index').mul(100).round(2)

print("\n--- Model Year Era vs. EV Type (Row %) ---")
print(era_pct)


--- Model Year Era vs. EV Type (Row %) ---
EV_Type                Battery Electric Vehicle (BEV)  \
Model_Year                                              
Early (1999-2015)                               61.34   
Growth (2016-2019)                              68.92   
Expansion (2020-2022)                           81.73   
Current (2023-2027)                             83.89   

EV_Type                Plug-in Hybrid Electric Vehicle (PHEV)  
Model_Year                                                     
Early (1999-2015)                                       38.66  
Growth (2016-2019)                                      31.08  
Expansion (2020-2022)                                   18.27  
Current (2023-2027)                                     16.11  


**Observation / Report (Categorical vs. Target):**
* **CAFV Qualification Split:** 100% of vehicles with *"Eligibility unknown as battery range has not been researched"* are **BEVs**, whereas *"Not eligible due to low battery range"* consists almost entirely of **PHEVs** whose electric-only range falls below the 30-mile clean fuel threshold.
* **BEV Share Over Time:** BEV share has expanded steadily from early adoption years to over **80%+** in the **Current (2023–2027)** model year bracket.

In [49]:
# Select numeric columns for comparison across EV_Type
num_cols = ['Model_Year', 'Vehicle_Age', 'Electric_Range']

# Comparing Mean and Median across EV_Type (BEV vs. PHEV)
bivariate_num = df.groupby('EV_Type', observed=False)[num_cols].agg(['mean', 'median']).T.round(2)
bivariate_num

EV_Type                Battery Electric Vehicle (BEV)  \
Model_Year     mean                           2022.62   
               median                         2023.00   
Vehicle_Age    mean                              4.38   
               median                            4.00   
Electric_Range mean                             37.08   
               median                            0.00   

EV_Type                Plug-in Hybrid Electric Vehicle (PHEV)  
Model_Year     mean                                   2021.43  
               median                                 2023.00  
Vehicle_Age    mean                                      5.57  
               median                                    4.00  
Electric_Range mean                                     32.26  
               median                                   32.00

**Observation / Report (Numerical vs. Target):**
* **Median Separation:** While both BEVs and PHEVs share recent median model years, their median `Electric_Range` differs sharply because newer BEVs carry unresearched `0`-mile placeholders while PHEVs retain explicit battery range figures (~25–35 miles).

In [50]:
# Extracting the Strongest Bivariate Feature Correlations
corr_matrix = df[['Model_Year', 'Vehicle_Age', 'Electric_Range', 'Legislative_District']].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
top_pairs = upper_tri.unstack().dropna().sort_values(ascending=False).head(6).round(2)

print("--- Top Strongest Bivariate Feature Relationships ---")
top_pairs.to_frame(name='Correlation Strength')

--- Top Strongest Bivariate Feature Relationships ---


Correlation Strength
Vehicle_Age          Model_Year                      1.00
Electric_Range       Model_Year                      0.55
                     Vehicle_Age                     0.55
Legislative_District Electric_Range                  0.00
                     Vehicle_Age                     0.00
                     Model_Year                      0.00

**Observation / Report (Numerical vs. Numerical):**
* **Model Year & Range Coupling:** Aside from the exact structural relationship between `Model_Year` and `Vehicle_Age` (`1.00`), the strongest numeric relationship is between `Model_Year` (or `Vehicle_Age`) and `Electric_Range` (`~0.51`), driven by the shift in EPA range reporting for post-2020 BEVs.

### Missing Value Analysis

Verifying the presence and percentage of missing (`Null`) values across all columns in the cleaned dataset to confirm data completeness prior to further analysis or modeling.

In [51]:
# Checking total and percentage of missing values in each column
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing Percentage (%)': missing_pct.round(2)
})
missing_df

,Missing Count,Missing Percentage (%)
VIN_Prefix,0,0.0
County,0,0.0
City,0,0.0
State,0,0.0
Postal_Code,0,0.0
Model_Year,0,0.0
Make,0,0.0
Model,0,0.0
EV_Type,0,0.0
CAFV_Eligibility,0,0.0


<span style="color: #f5f2f2; font-weight: bold; font-style: italic; font-size: 20px;">
  Observation / Report (Missing Value Analysis):
</span>

* **Zero Missing Values:** All **18 columns** across the **300,142 vehicle records** show a **0.0% missing rate**, confirming that group-median range imputation (`Electric_Range`), numeric zero-filling (`Legislative_District`, `Postal_Code`, `Census_Tract_2020`), and `'Unknown'` categorical labeling successfully resolved all incomplete entries.